# 🧪 PipeScraper Tutorial: Modern News Scraping

Welcome to **PipeScraper**! This tutorial will guide you through the core concepts of the library, from simple extractions to building advanced analysis pipelines using the pipe (`>>`) operator.

## 🚀 1. Installation

PipeScraper is designed to be lightweight but powerful. You can install it and its primary integrations using pip:

In [ ]:
# !pip install pipescraper pipeframe pipeplotly

## 🛠️ 2. The Basic Pipeline

PipeScraper uses a natural language syntax. Let's scrape some news from the BBC.

In [1]:
from pipescraper import FetchLinks, ExtractArticles, ToDataFrame

url = "https://www.bbc.com/news"

# building a pipeline: Fetch -> Extract -> DataFrame
df = (url >> 
      FetchLinks(max_links=5) >> 
      ExtractArticles() >> 
      ToDataFrame())

df.head()

,url,source,title,text,description,author,date_published,time_published,language,tags,image_url
0,https://www.bbc.com/news/articles/cn87ndvklr2o,www.bbc.com,"Hozier, Jessie Buckley and Bruce Springsteen r...","Hozier, Jessie Buckley and Bruce Springsteen r...","The album will also feature Primal Scream, Dav...",Mark Savage,2026-03-12,,,[],https://ichef.bbci.co.uk/news/1024/branded_new...
1,https://www.bbc.com/business/world-of-business,www.bbc.com,BBC Business | World of Business,World of BusinessSpain's migrants welcome amne...,,,2025-03-21,,,[],
2,https://www.bbc.com/news/world/africa,www.bbc.com,Africa | Latest News & Updates | BBC News,NewsAfricaSenegal approves tougher anti-gay la...,"Get all the latest news, live updates and cont...",,2026-01-01,,,[],
3,https://www.bbc.com/technology/ai-v-the-mind,www.bbc.com,BBC Technology | AI v the Mind,AI v the MindMeet the world's first recipient ...,,,2024-08-23,,,[],
4,https://www.bbc.com/news/world/australia,www.bbc.com,Australia | Latest News & Updates | BBC News,NewsAustraliaBodies of two Chinese backpackers...,"Get all the latest news, live updates and cont...",,2026-02-27,,,[],


## ⏱️ 3. Performance & Time Extraction

PipeScraper is optimized for speed. By default, it extracts content and publication dates very fast. If you need the specific **time** of publication, you can enable `extract_time=True`.

## ⚡ 4. Parallel Scraping (Turbo Mode)

Scraping many articles sequentially can be slow. PipeScraper v0.2.1 introduces **Parallel Scraping** using the `workers` parameter. This can speed up your pipelines by up to 10x!

In [16]:
import time
from pipescraper import FetchLinks, ExtractArticles

start = time.time()
# Use workers=10 to scrape articles in parallel
articles = (url 
            >> FetchLinks(max_links=10)  
            >> ExtractArticles(workers=10)
            >> ToDataFrame())
end = time.time()

print(f"Turbo Scraped {len(articles)} articles in {end - start:.2f} seconds!")

Turbo Scraped 10 articles in 2.98 seconds!


In [17]:
df.head()

,url,source,title,text,description,author,date_published,time_published,language,tags,image_url
0,https://www.bbc.com/news/northern_ireland,www.bbc.com,Northern Ireland | Latest News & Updates | BBC...,NewsNewsN. IrelandN. Ireland Politics'Most of ...,"Get all the latest news, live updates and cont...",,2026-01-01,,,,
1,https://www.bbc.com/sport/articles/c98gkg3kr1jo,www.bbc.com,Winter Paralympics 2026: Russia and Belarus at...,Russian athletes march at Paralympic opening c...,Athletes from Russia and Belarus parade behind...,Katie Falkingham,2026-03-06,,,,https://ichef.bbci.co.uk/ace/branded_sport/120...
2,https://www.bbc.com/sport/football/articles/ce...,www.bbc.com,World Cup 2026: Co-hosts Mexico plan to deploy...,"Mexico to deploy 100,000 security personnel fo...",World Cup co-hosts Mexico plan to deploy nearl...,Harry Poole,2026-03-06,,,,https://ichef.bbci.co.uk/ace/branded_sport/120...
3,https://www.bbc.com/news/articles/c14m57k8vgeo,www.bbc.com,Surge in jet fuel prices could push up air far...,Surge in jet fuel prices could push up air far...,Disruption to supplies from the Gulf due to th...,Theo Leggett,2026-03-06,,,,https://ichef.bbci.co.uk/news/1024/branded_new...
4,https://www.bbc.com/arts/arts-in-motion,www.bbc.com,Arts in Motion,Arts in Motion In a pioneering new collaborati...,,,2026-01-16,,,,


In [18]:
# With time extraction enabled (slightly slower as it uses Newspaper3k supplement)
articles = (url >> 
            FetchLinks(max_links=3) >> 
            ExtractArticles(extract_time=True))

for a in articles:
    # Note: .to_dict() is available on Article objects
    data = a.to_dict()
    print(f"Title: {data['title']}")
    print(f"Time: {data['time_published']}\n")

Optional newspaper3k extraction failed for https://www.bbc.com/sport/articles/c98gkg3kr1jo: 'Article' object has no attribute 'set_html'
Optional newspaper3k extraction failed for https://www.bbc.com/news/northern_ireland: 'Article' object has no attribute 'set_html'
Optional newspaper3k extraction failed for https://www.bbc.com/sport/football/articles/ce8wjwyjndyo: 'Article' object has no attribute 'set_html'


Title: Northern Ireland | Latest News & Updates | BBC News
Time: 

Title: Winter Paralympics 2026: Russia and Belarus athletes parade behind national flags in opening ceremony
Time: 

Title: World Cup 2026: Co-hosts Mexico plan to deploy 100,000 security personnel
Time: 



In [19]:
articles

[Article(url='https://www.bbc.com/news/northern_ireland', source='www.bbc.com', title='Northern Ireland | Latest News & Updates | BBC News', text='NewsNewsN. IrelandN. Ireland Politics\'Most of my pension has gone on home heating oil\'Rising heating oil prices are hitting Northern Ireland harder than the rest of the UK - here\'s everything you need to know.1 hr agoNorthern IrelandNatalie McNally murder accused beat ex-partner, court hearsNatalie McNally was 15 weeks pregnant when she died at her home in Lurgan in December 2022.9 hrs agoNorthern IrelandBelfast\'s QFT named \'one of the greatest cinemas in the world\'Queen\'s Film Theatre has been ranked as number 22 on a list of "the greatest cinemas in the world right now".8 hrs agoNorthern IrelandOwner\'s relief as social media appeal finds stolen carChristian Van Der Merwe\'s car was one of three stolen in Derry\'s Waterside earlier this week.11 hrs agoNorthern IrelandTwo of the world\'s rarest lions put to sleep at zooThheiba and Fi

## 🛡️ 5. Robust Utilities: Deduplicate, Limit, and Delay

When crawling larger sites, you want to be respectful and avoid duplicate data.

In [26]:
from pipescraper import Deduplicate, LimitArticles, WithDelay

processed_articles = (url >> 
                      FetchLinks(max_links=20) >> 
                    #   Deduplicate() >> 
                      LimitArticles(5) >> 
                      WithDelay(0.5) >>    # 0.5s delay between article requests
                      ExtractArticles())

## 🔍 6. Advanced Filtering

Filter your articles list based on any custom condition before converting to a DataFrame.

In [27]:
from pipescraper import FilterArticles

filtered = (processed_articles >> 
            FilterArticles(lambda a: "Sport" in a.text or "Football" in a.title))

print(f"Original: {len(processed_articles)}")
print(f"Filtered: {len(filtered)}")

Original: 5
Filtered: 1


## 📂 7. Exporting Data

You can easily save your results to various formats.

In [ ]:
from pipescraper import SaveAs

df >> SaveAs("my_news.csv")
df >> SaveAs("my_news.json")
df >> SaveAs("my_news.parquet")

## 🔗 8. Advanced Analysis with PipeFrame

PipeScraper integrates with `pipeframe` for tidy data manipulation. Move from raw articles to clean insights in seconds.

In [32]:
from pipescraper import ToPipeFrame, select, mutate, filter_df, group_by, summarize, arrange, pf_head

pf_result = (url >> 
             FetchLinks(max_links=15) >> 
             ExtractArticles() >> 
             ToPipeFrame() >>
             # Selecting specific columns
             select('source', 'title', 'author', 'date_published') >>
             # Filtering out rows with no author
            #  filter_df(lambda df: df['author'] != '') >>
             # Grouping by source and counting
            #  group_by('source') >>
            #  summarize(count=lambda df: len(df)) >>
             # Arranging results
            #  arrange('count', ascending=False) >>
             pf_head(10))

pf_result

<pipeframe.DataFrame shape=(10, 4)>
        source                                              title  \
0  www.bbc.com  Northern Ireland | Latest News & Updates | BBC...   
1  www.bbc.com  Winter Paralympics 2026: Russia and Belarus at...   
2  www.bbc.com  World Cup 2026: Co-hosts Mexico plan to deploy...   
3  www.bbc.com  Surge in jet fuel prices could push up air far...   
4  www.bbc.com                                     Arts in Motion   
5  www.bbc.com                            BBC Travel | Adventures   
6  www.bbc.com  BBC Travel | Destinations | Australia and Pacific   
7  www.bbc.com            BBC Live & Breaking World and U.S. News   
8  www.bbc.com  Withheld Jeffrey Epstein files with accusation...   
9  www.bbc.com                 BBC Travel | Culture & Experiences   

             author date_published  
0                       2026-01-01  
1  Katie Falkingham     2026-03-06  
2       Harry Poole     2026-03-06  
3      Theo Leggett     2026-03-06  
4                  

## 📊 9. Visualization with PipePlotly

Visualize your scraped data using `ggplot` syntax or pre-built chart functions.

In [34]:
from pipescraper import ggplot, aes, geom_bar, geom_line, geom_point, geom_histogram, labs, show, set_theme

# Let's create a more complex chart
chart = (pf_result >> 
         ggplot(aes(x='source', fill='source')) >> 
         geom_bar() >> 
         labs(title="Articles by Source", x="Source", y="Count") >>
         set_theme("minimal"))

if chart:
    show(chart)

PipeFrameTypeError: Pipe operator '>>' requires callable, got Plot

### 🖼️ 10. Pre-built Chart Functions

For even faster visualization, use these specialized functions:

In [35]:
from pipescraper import create_articles_by_source_chart, create_text_length_distribution, create_articles_timeline

df = url >> FetchLinks(max_links=10) >> ExtractArticles() >> ToDataFrame()

# 1. Articles by Source
fig1 = create_articles_by_source_chart(df)

# 2. Text Length Distribution
fig2 = create_text_length_distribution(df)

# 3. Timeline of Publication
fig3 = create_articles_timeline(df)

show(fig1)
show(fig2)
show(fig3)

ImportError: cannot import name 'ggplot' from 'pipeplotly' (C:\Users\yasse\AppData\Roaming\Python\Python311\site-packages\pipeplotly\__init__.py)

## 🎯 Summary

You've learned how to:
1. Build scraping pipelines.
2. Use core classes (`Article`, `LinkFetcher`, `ArticleExtractor`).
3. Apply utilities like `Deduplicate`, `LimitArticles`, and `WithDelay`.
4. Perform advanced filtering with `FilterArticles`.
5. Export data to multiple formats with `SaveAs`.
6. Manipulate data with `PipeFrame` (grouping, summarizing).
7. Visualize data with `PipePlotly` and pre-built chart functions.

Explore the **API Reference** for more details on all available parameters and verbs!